# 04. Case Reuse & Outcome Prediction

This notebook implements the **Reuse** phase of the Case-Based Reasoning (CBR) cycle. In this phase, the retrieved cases are used to solve the new query case.

### Objectives:
1. **Load System Artifacts**: Load the cases database (`data/processed/cases.csv`) and the fitted TF-IDF Vectorizer (`models/tfidf_vectorizer.pkl`).
2. **Implement Retrieval Logic**: Reconstruct the Cosine Similarity retrieval helper.
3. **Outcome Prediction via Majority Voting**: Implement `predict_outcome(query)` using the Top-5 retrieved cases. If a tie occurs, we use **similarity-weighted voting** (sum of similarity scores per outcome class) to resolve it.
4. **Adaptation**: Reuse and suggest the verdict details (Amar Putusan) from the most similar matching case.
5. **Validation & Batch Run**: Run the prediction system on a set of typical legal query scenarios and save the predictions to `data/processed/predictions.csv`.

In [8]:
import os
import re
import pickle
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics.pairwise import cosine_similarity

# Import Sastrawi components for query preprocessing
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

In [9]:
# Define paths and load cases
CASES_CSV_PATH = Path("../data/processed/cases.csv")
VECTORIZER_PATH = Path("../models/tfidf_vectorizer.pkl")
OUTPUT_PREDICTIONS_PATH = Path("../data/processed/predictions.csv")

df_cases = pd.read_csv(CASES_CSV_PATH)
with open(VECTORIZER_PATH, 'rb') as f:
    vectorizer = pickle.load(f)

# Reconstruct full text representing each case
RAW_TEXT_DIR = Path("../data/raw_text")
texts = []
for _, row in df_cases.iterrows():
    case_id = row['Case ID']
    txt_file = RAW_TEXT_DIR / f"{case_id}.txt"
    if txt_file.exists():
        with open(txt_file, 'r', encoding='utf-8') as f:
            texts.append(f.read())
    else:
        combined_meta = f"{row['Ringkasan fakta']} {row['Amar putusan']} {row['Pasal']}"
        texts.append(combined_meta)

df_cases['cleaned_text'] = texts

# Fit/Transform full matrix
tfidf_matrix = vectorizer.transform(df_cases['cleaned_text'])
print(f"Loaded {len(df_cases)} cases. TF-IDF Matrix shape: {tfidf_matrix.shape}")

Loaded 30 cases. TF-IDF Matrix shape: (30, 1000)


In [10]:
# Define binary outcome labels (Verdict: Bersalah vs Bebas/Lepas)
def extract_verdict_class(amar_text):
    amar_lower = str(amar_text).lower()
    if "bebas" in amar_lower or "lepas" in amar_lower or "onslag" in amar_lower or "tidak terbukti" in amar_lower:
        return "Bebas/Lepas"
    return "Bersalah"

df_cases['Verdict_Class'] = df_cases['Amar putusan'].apply(extract_verdict_class)
print(df_cases['Verdict_Class'].value_counts())

Verdict_Class
Bersalah       27
Bebas/Lepas     3
Name: count, dtype: int64


In [11]:
# Preprocessing components
stop_factory = StopWordRemoverFactory()
indonesian_stopwords = set(stop_factory.get_stop_words())
indonesian_stopwords.update({'dan', 'yang', 'untuk', 'dengan', 'pada', 'atau', 'adalah', 'bahwa'})

stem_factory = StemmerFactory()
stemmer = stem_factory.create_stemmer()

def preprocess_query(query: str) -> str:
    """Cleans, tokenizes, filters and stems the query."""
    query_clean = re.sub(r'[^a-zA-Z\s]', ' ', query.lower())
    tokens = query_clean.split()
    filtered_tokens = [w for w in tokens if w not in indonesian_stopwords]
    stemmed = [stemmer.stem(w) for w in filtered_tokens]
    return ' '.join([w for w in stemmed if len(w) > 1])

def retrieve(query: str, k: int = 5) -> pd.DataFrame:
    """Retrieves top-k cases using cosine similarity."""
    cleaned_query = preprocess_query(query)
    query_vector = vectorizer.transform([cleaned_query])
    similarities = cosine_similarity(query_vector, tfidf_matrix).flatten()
    
    df_res = df_cases.copy()
    df_res['Similarity Score'] = similarities
    return df_res.sort_values(by='Similarity Score', ascending=False).head(k)

In [12]:
def predict_outcome(query: str) -> dict:
    """
    Retrieves top-5 cases and uses similarity-weighted majority voting to predict
    whether the outcome is Conviction (Bersalah) or Acquittal (Bebas/Lepas).
    Adapts/suggests the sentencing or verdict based on the closest match.
    """
    top_cases = retrieve(query, k=5)
    
    # Weighted Voting based on Cosine Similarity
    votes = {}
    for _, row in top_cases.iterrows():
        outcome = row['Verdict_Class']
        score = row['Similarity Score']
        # Add score as weight; if similarity is 0, give a tiny epsilon weight to avoid ignoring
        votes[outcome] = votes.get(outcome, 0.0) + max(score, 0.01)
        
    # Get majority predicted class
    predicted_outcome = max(votes, key=votes.get)
    
    # Get the best match details for adaptation
    best_match = top_cases.iloc[0]
    
    return {
        "Query": query,
        "Predicted Outcome": predicted_outcome,
        "Top Match Case ID": best_match['Case ID'],
        "Top Match Score": round(best_match['Similarity Score'], 4),
        "Top Match Docket": best_match['Nomor perkara'],
        "Top Match Defendant": best_match['Nama terdakwa'],
        "Top Match Pasal": best_match['Pasal'],
        "Recommended Verdict (Amar)": best_match['Amar putusan']
    }

In [13]:
# Define testing legal scenarios for outcome prediction
scenarios = [
    "terdakwa memalsukan tanda tangan pemilik tanah ahli waris pada surat kuasa jual beli sehingga mengakibatkan kerugian bagi korban",
    "terdakwa didakwa menggunakan surat palsu berupa kuitansi pembayaran palsu untuk menggelapkan uang perusahaan",
    "tanda tangan dalam surat pernyataan kesepakatan ganti rugi tanah dinyatakan non-identik oleh laboratorium kriminalistik kriminal",
    "dakwaan jaksa penuntut umum kabur dan dibebaskan demi hukum oleh majelis hakim karena tidak terbukti melakukan tindak pidana pemalsuan",
    "terdakwa secara bersama-sama memasang spanduk klaim kepemilikan tanah dan memalsukan surat keterangan tanah di kantor kelurahan"
]

prediction_results = []

for scenario in scenarios:
    pred = predict_outcome(scenario)
    prediction_results.append(pred)

df_predictions = pd.DataFrame(prediction_results)

# Display predictions
display(df_predictions[[
    'Query', 
    'Predicted Outcome', 
    'Top Match Case ID', 
    'Top Match Score', 
    'Top Match Pasal'
]])

,Query,Predicted Outcome,Top Match Case ID,Top Match Score,Top Match Pasal
0,terdakwa memalsukan tanda tangan pemilik tanah...,Bersalah,case009,0.2773,"Pasal 263 Ayat (1) KUHP, pasal 263 ayat (1) Ki..."
1,terdakwa didakwa menggunakan surat palsu berup...,Bersalah,case006,0.3191,Pasal 264 ayat (2) KUHP
2,tanda tangan dalam surat pernyataan kesepakata...,Bersalah,case022,0.2368,"Pasal 263 ayat (2) KUHP, Pasal 263 ayat (2) KUHp"
3,dakwaan jaksa penuntut umum kabur dan dibebask...,Bersalah,case018,0.2723,Pasal 263 KUHP
4,terdakwa secara bersama-sama memasang spanduk ...,Bersalah,case022,0.3510,"Pasal 263 ayat (2) KUHP, Pasal 263 ayat (2) KUHp"


In [14]:
# Save predictions
df_predictions.to_csv(OUTPUT_PREDICTIONS_PATH, index=False, encoding='utf-8')
print(f"Predictions saved successfully to {OUTPUT_PREDICTIONS_PATH.resolve()}")

Predictions saved successfully to D:\Lectures\Semester 8\Penalaran\CBR-Pemalsuan-Surat\data\processed\predictions.csv
